# 06 · Evaluate the current Transformer

Evaluate held-out predictions from the current Transformer once trained, measuring how it prioritizes held-out `RESP=1` snapshots using a detailed decile table, cumulative gains, lift and top-10%/20%/30% capture and precision. LightGBM prediction and comparison are deferred.

**Inputs:** the original full frozen patient-level split manifest and one Transformer probability per TEST snapshot, either as CSV/Parquet files or existing in-memory pandas DataFrames. **Outputs:** aggregate tables, charts, a Markdown report and an audit record. There are no model results until this notebook runs against the work laptop's actual artifacts. The shared project reference describes a tensor and proposed model, but does not confirm a saved checkpoint or completed training. Confirm training status and use the original frozen split and matching predictions on that laptop.

**Confirmed project context:** use historical V63 and claims vintage `20260825`; predict first advanced-therapy initiation within 90 days. The sample is `PATIENT_ID + END_DT`. A patient can contribute multiple snapshots but must never cross TRAIN/VALIDATION/TEST splits. This notebook checks the supplied manifest and scores; it cannot establish training history, claims lineage or absence of upstream leakage from those files alone.

Keep this notebook beside `targeting_evaluation.py`. Install `requirements.txt` in the notebook kernel environment. Do not regenerate the split, rebuild the population, alter the natural TEST class balance or tune the model after inspecting TEST results.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from targeting_evaluation import (
    evaluate_model,
    manifest_fingerprint,
    read_input,
    validate_inputs,
    validate_manifest,
    validate_provenance,
    write_report,
)

## 1. Select existing inputs

File mode reads `TAK861_SPLIT_MANIFEST` and `TAK861_TRANSFORMER_PREDICTIONS` from the kernel environment. `TAK861_PROVENANCE` is optional. Set `TAK861_EVALUATION_OUTPUT` to change the default local artifact directory.

For in-memory mode, set `USE_IN_MEMORY = True` and first create `frozen_snapshot_manifest` and `transformer_test_predictions` in this kernel using the existing training/inference pipeline. Both must be pandas DataFrames. An optional `transformer_run_metadata` dictionary uses the provenance schema in the README. This notebook never prints the source DataFrames or their paths.

Manifest columns: `PATIENT_ID` (string), `END_DT` (`YYYY-MM-DD`), `RESP` (0/1), `SPLIT` (`TRAIN`, `VALIDATION`, `TEST`). Predictions: `PATIENT_ID`, `END_DT`, `P_RESP1`. Predictions must contain exactly the TEST snapshots; optional `RESP`/`SPLIT` columns are cross-checked.

In [ ]:
USE_IN_MEMORY = False
OUTPUT_DIR = Path(os.environ.get(
    "TAK861_EVALUATION_OUTPUT", "artifacts/transformer_evaluation"
)).expanduser()

def required_input(variable_name):
    value = os.environ.get(variable_name, "").strip()
    if not value:
        raise RuntimeError(
            f"Set {variable_name} to an existing local CSV/Parquet input, "
            "or configure the documented in-memory mode."
        )
    return read_input(Path(value).expanduser())

if USE_IN_MEMORY:
    required_names = ("frozen_snapshot_manifest", "transformer_test_predictions")
    if any(name not in globals() for name in required_names):
        raise RuntimeError("Create both documented in-memory input DataFrames first.")
    if not all(isinstance(globals()[name], pd.DataFrame) for name in required_names):
        raise TypeError("In-memory inputs must be pandas DataFrames.")
    manifest_input = frozen_snapshot_manifest.copy()
    predictions_input = transformer_test_predictions.copy()
    run_metadata = globals().get("transformer_run_metadata")
else:
    manifest_input = required_input("TAK861_SPLIT_MANIFEST")
    predictions_input = required_input("TAK861_TRANSFORMER_PREDICTIONS")
    provenance_path = os.environ.get("TAK861_PROVENANCE", "").strip()
    run_metadata = None
    if provenance_path:
        with Path(provenance_path).expanduser().open(encoding="utf-8") as handle:
            run_metadata = json.load(handle)

### Connecting current model inference

Use existing score exports when available. Otherwise keep keys and scores together throughout the trained model's inference pipeline. For a binary-logit output, use the existing model in evaluation mode without gradient tracking, then apply sigmoid exactly once. Preserve the known positive-class meaning. The evaluation package requires no PyTorch dependency because it consumes probabilities.

The following is an adapter to use **after** the existing inference code; it does not load a checkpoint or infer an architecture:

```python
# Keys and probabilities must come from the SAME ordered inference records.
probabilities = np.asarray(test_probabilities)
if probabilities.ndim == 2 and probabilities.shape[1] == 1:
    probabilities = probabilities[:, 0]
if probabilities.ndim != 1 or len(test_snapshot_keys) != len(probabilities):
    raise ValueError("Expected one probability per matching snapshot key.")
transformer_test_predictions = test_snapshot_keys[["PATIENT_ID", "END_DT"]].copy()
transformer_test_predictions["P_RESP1"] = probabilities
```

Length equality alone cannot prove correspondence: verify the existing inference order, especially with shuffled loaders. Do not independently sort the keys or flatten a two-class probability matrix. Refer to the README for the complete adapter. Keep any patient-level objects local.

## 2. Validate the frozen split and score coverage

Validation fails for duplicate keys, nonbinary/missing labels, patient overlap between splits, invalid probabilities, or missing/extra TEST predictions. Scores are aligned by both snapshot keys. Optional prediction labels must match the canonical manifest labels.

The fingerprint records the exact full manifest, including labels and frozen assignments, independently of input order. An optional provenance declaration must match that fingerprint, V63, vintage `20260825` and the intended split roles. These declarations do not audit the training code. Missing provenance remains explicitly unverified.

In [ ]:
manifest = validate_manifest(manifest_input)
aligned = validate_inputs(manifest, predictions_input)
manifest_sha256 = manifest_fingerprint(manifest)

if run_metadata is None:
    provenance_status = "Not supplied; upstream model/split/vintage history is unverified."
else:
    validate_provenance(run_metadata, manifest)
    provenance_status = "Supplied declarations match; training code was not independently audited."

audit = {
    "snapshot_manifest_sha256": manifest_sha256,
    "provenance_status": provenance_status,
    "patient_disjointness_checked": True,
    "exact_test_snapshot_match_checked": True,
    "training_code_audited": False,
}

# Aggregate checks only: never display manifest_input, predictions_input or aligned.
split_summary = manifest.groupby("SPLIT", sort=False).agg(
    n_snapshots=("RESP", "size"),
    n_patients=("PATIENT_ID", "nunique"),
    n_resp1=("RESP", "sum"),
).reindex(["TRAIN", "VALIDATION", "TEST"])
split_summary["response_rate"] = split_summary.n_resp1 / split_summary.n_snapshots
display(split_summary)
display(Markdown("**Provenance:** " + provenance_status))

## 3. Rank without labels and evaluate the ten deciles

Rank TEST snapshots by descending `P_RESP1`; decile 1 is highest propensity and decile 10 is lowest. Resolve equal scores by a fixed SHA-256 hash of the canonical snapshot identity, without consulting `RESP`. For `N` TEST snapshots, cumulative decile boundary `d` is `ceil(d*N/10)`. All ten buckets are nonempty when `N >= 10`, and their sizes differ by at most one.

Labels enter only after ranks and deciles have been fixed. Top-10%/20%/30% metrics reuse the first one/two/three complete deciles. The exact selected count and fraction are reported, including rounding when `N` is not divisible by ten.

In [ ]:
results = evaluate_model(aligned)
assert len(results["deciles"]) == 10
assert set(results["topk"]["top_k_pct"]) == {10, 20, 30}
assert results["deciles"]["n_snapshots"].sum() == len(aligned)
assert results["deciles"]["n_resp1"].sum() == int(aligned["RESP"].sum())

def present_aggregates(frame):
    """Render fractions as percentages, preserving raw numeric result tables."""
    table = frame.copy()
    rate_columns = {
        "population_fraction", "response_rate", "decile_precision",
        "share_of_all_resp1", "cumulative_population_fraction",
        "cumulative_recall", "cumulative_precision", "overall_test_response_rate",
        "actual_population_fraction", "recall", "precision", "test_response_rate",
    }
    lift_columns = {"decile_lift", "cumulative_lift", "lift"}
    for column in rate_columns.intersection(table.columns):
        table[column] = table[column].map(lambda value: "N/A" if pd.isna(value) else f"{value:.2%}")
    for column in lift_columns.intersection(table.columns):
        table[column] = table[column].map(lambda value: "N/A" if pd.isna(value) else f"{value:.2f}x")
    return table

display(present_aggregates(results["deciles"]))

**Reading the table:** `share_of_all_resp1` is the percentage of **all TEST positives** in one decile. `cumulative_resp1` is the cumulative positive count, and `cumulative_recall` is cumulative capture. `response_rate` and `decile_precision` both describe the current bucket. `cumulative_precision` describes the entire selected population through that bucket. Lift divides the relevant response rate by the overall TEST response rate.

The numeric CSV tables store proportions on a 0–1 scale; the display above formats them as percentages. See `EVALUATION_SPEC.md` for every formula. No unique-patient deduplication occurs.

## 4. Explicit top-K capture, precision and lift

These rows report Recall@10%, Recall@20%, Recall@30%, Precision@10%, Precision@20%, Precision@30% and lift at the same capacities. The top-decile lift is the first row's lift. Each row states how many snapshots and actual positives are selected.

In [ ]:
display(present_aggregates(results["topk"]))
first_decile_lift = results["deciles"].loc[
    results["deciles"]["decile"].eq(1), "decile_lift"
].iloc[0]
lift_text = "N/A" if pd.isna(first_decile_lift) else f"{first_decile_lift:.3f}x"
display(Markdown("**Top-decile lift:** " + lift_text))

## 5. Supporting global metrics and score ties

Average precision summarizes the precision-recall tradeoff using scikit-learn's non-interpolated definition; it is not trapezoidal PR-AUC ([official documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html)). ROC-AUC supplies additional global discrimination context. Neither replaces top-K targeting performance.

A tie spanning a selected cutoff means some equally scored snapshots are selected and others are not. The deterministic key hash makes this reproducible but contributes no predictive information. With no TEST positives, recall/capture, lift and average precision are N/A. ROC-AUC is N/A when either class is absent.

In [ ]:
display(present_aggregates(results["global_metrics"]))
display(results["ties"])

## 6. Save aggregate reports and show the charts

The gains chart uses the actual cumulative selected fraction and a random-ranking diagonal. Lift charts compare individual-decile and cumulative lift with a baseline of 1. The top-K chart summarizes the current Transformer at the three review capacities.

Only aggregate tables, charts, a report and an audit record are written. The local default output directory is ignored by Git. Keep generated reports within your approved environment and clear notebook outputs before sharing code.

In [ ]:
report_paths = write_report(results, OUTPUT_DIR, audit=audit)
for report_name, report_path in report_paths.items():
    report_path = Path(report_path)
    if report_path.suffix.lower() == ".png":
        display(Image(filename=str(report_path)))

display(Markdown("**Generated aggregate artifacts:**\n\n" + "\n".join(
    "- `" + Path(path).name + "`" for path in report_paths.values()
)))

## 7. Interpretation and deferred comparison

Use the observed top-K capture, precision and lift to assess how concentrated actual positive snapshots are within a fixed review capacity. Compare gains with the random-ranking reference and examine whether ties make a chosen boundary sensitive to an arbitrary selection among equal scores.

These are snapshot-level point estimates. Multiple snapshots from one patient remain separate; no confidence interval or unique-patient outreach estimate is implied. Probability calibration and a validation-selected operating threshold require separate checks. This ranking-focused run does not choose a classification threshold from TEST labels or produce threshold-based F1/confusion-matrix claims.

**Transformer incremental value over LightGBM is not yet measured.** When the correct LightGBM artifact is available, evaluate it on this exact frozen manifest and TEST snapshot set, then compare captured positives, Recall@K, Precision@K and lift at equal selected sizes. Retain the manifest fingerprint and these definitions for that later comparison. A better global ROC-AUC alone would not establish better prioritization of the highest-propensity population.